# Wheat detection - training (Part 7)
Before running, in the right-hand **Session options**: Accelerator = **GPU T4 x2**, Internet = **On**,
and **Add Input** -> the *Global Wheat Detection* competition.
Run once with `SMOKE = True` (a few minutes), then set it to `False` and use **Save Version -> Save & Run All**.
Download the results from the version's **Output** tab.

In [1]:
import os
from pathlib import Path

if Path("../../.git").is_dir():
    os.chdir("./../..")

In [2]:
is_local = not os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
if is_local:
    print("Running locally (or on a non-Kaggle server).")
else:
    print("Running in a Kaggle Notebook environment.")

Running locally (or on a non-Kaggle server).


In [10]:
REPO = "Saatvik-Sinha/wheat-mlops"
BRANCH = "training"
SMOKE = False

In [11]:
if is_local:
    REPO_DIR = Path()
    DATA_DIR = Path("data")
else:
    DATA_DIR = Path("/kaggle")
    (DATA_DIR / "temp").mkdir(exist_ok=True)
    REPO_DIR = DATA_DIR / "temp/repo"

    token = None
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"

In [12]:
![ {is_local} == False ] && rm -rf {REPO_DIR}
![ {is_local} == False ] && git clone --depth 1 --branch {BRANCH} {url} {REPO_DIR}
![ {is_local} == False ] && pip install -q "ultralytics>=8.3,<9"

In [13]:
import json
import sys

import yaml

sys.path.insert(0, str(REPO_DIR / "src" / "python"))

from wheat_mlops.prepare_data import prepare


def find_raw() -> Path:
    for candidate in (
        DATA_DIR / "input/competitions/global-wheat-detection",  # As a competition
        DATA_DIR / "input/global-wheat-detection",  # As a dataset
        DATA_DIR / "raw",  # Locally
    ):
        if (candidate / "train.csv").is_file():
            return candidate
    raise FileNotFoundError("Competition data not found - use Add Input -> Global Wheat Detection")


RAW = find_raw()
PROCESSED = DATA_DIR / "processed"
p = yaml.safe_load((REPO_DIR / "params.yaml").read_text(encoding="utf-8"))["data"]
stats = prepare(
    raw_dir=RAW,
    out_dir=PROCESSED,
    val_fraction=p["val_fraction"],
    seed=p["seed"],
    count_bucket=p["count_bucket"],
    max_count_bucket=p["max_count_bucket"],
    min_box_area=p["min_box_area"],
    max_box_area=p["max_box_area"],
    link_images=False,  # /kaggle/input is a different filesystem, so copy
)
print(f"raw data: {RAW}")
print(json.dumps(stats, indent=2))

raw data: data\raw
{
  "n_images": 3422,
  "n_train": 2737,
  "n_val": 685,
  "n_background_images": 49,
  "n_boxes_raw": 147793,
  "n_boxes_kept": 147749,
  "n_boxes_dropped": 44,
  "boxes_per_image_mean": 43.18,
  "images_per_source": {
    "train": {
      "arvalis_1": 844,
      "ethz_1": 597,
      "arvalis_3": 447,
      "rres_1": 345,
      "arvalis_2": 164,
      "usask_1": 160,
      "inrae_1": 141,
      "unknown": 39
    },
    "val": {
      "arvalis_1": 211,
      "ethz_1": 150,
      "arvalis_3": 112,
      "rres_1": 87,
      "usask_1": 40,
      "arvalis_2": 40,
      "inrae_1": 35,
      "unknown": 10
    }
  }
}


In [14]:
OUT_DIR = DATA_DIR / ("temp/trained_smoke" if SMOKE else "working/trained")

In [12]:
import sys

argv = [
    f"--data-yaml {PROCESSED}/dataset.yaml",
    f"--out-dir {OUT_DIR}",
    f"--params {REPO_DIR}/train_params.yaml",
]
if SMOKE:
    argv.append("--smoke")

CMD = f"python {REPO_DIR}/src/python/wheat_mlops/train.py {' '.join(argv)}"
!{CMD}

^C


In [15]:
import yaml

with open(f"{REPO_DIR}/train_params.yaml", encoding="utf-8") as fin:
    cfg = yaml.safe_load(fin)
    cfg["train"]["model"] = str(DATA_DIR / "working/trained/best.pt")
    cfg["train"]["args"]["epochs"] = 200
    with open(f"{REPO_DIR}/improve_params.yaml", "w", encoding="utf-8") as fout:
        yaml.safe_dump(cfg, fout)

argv = [
    f"--data-yaml {PROCESSED}/dataset.yaml",
    f"--out-dir {OUT_DIR}",
    f"--params {REPO_DIR}/improve_params.yaml",
]
if SMOKE:
    argv.append("--smoke")

# CMD = f"python {REPO_DIR}/src/python/wheat_mlops/train.py {" ".join(argv)}"
# !{CMD}

In [17]:
str(DATA_DIR / "working/trained/best.pt")

'data\\working\\trained\\best.pt'

In [16]:
argv

['--data-yaml data\\processed/dataset.yaml',
 '--out-dir data\\working\\trained',
 '--params ./improve_params.yaml']

In [ ]:
print(Path(OUT_DIR, "metrics.json").read_text())
!ls -la {OUT_DIR}